# Q-ROVER 3차시 — YOLOX-Nano 추가 학습
런타임 → 런타임 유형 변경 → **T4 GPU** 로 바꾼 뒤 위에서부터 실행

## 1. 데이터 (Drive의 `qrover/qrover_data.zip`)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!unzip -q -o "/content/drive/MyDrive/qrover/qrover_data.zip" -d /content/
!ls /content/qrover_data        # images  labels  labels.txt 가 보여야 합니다

## 2. YOLOX 받기

In [ ]:
%cd /content
!git clone -q https://github.com/Megvii-BaseDetection/YOLOX.git
%cd /content/YOLOX
!pip install -q loguru thop tabulate pycocotools onnx onnxruntime
!wget -q https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_nano.pth

import os, sys
os.environ["PYTHONPATH"] = "/content/YOLOX"
sys.path.insert(0, "/content/YOLOX")
!python -c "import os, yolox, torch; print('YOLOX', yolox.__version__, '| GPU', torch.cuda.is_available(), '| nano.pth', os.path.getsize('yolox_nano.pth') // 2**20, 'MB')"

## 3. 데이터 변환 (YOLO → COCO, 세션 단위 분할, 416 squash)

In [ ]:
%%writefile /content/prepare_dataset.py
"""팀 데이터(YOLO 형식) → YOLOX 학습용(COCO 형식)

입력 (SRC)
  images/s01_0001.jpg ...      파일 이름 앞부분(_ 앞)이 촬영 세션
  labels/s01_0001.txt ...      YOLO 형식: 클래스번호 cx cy w h (0~1)
  labels.txt                   클래스 이름, 한 줄에 하나. 순서 = 클래스 번호
출력 (DST)
  train2017/, val2017/         416x416으로 늘려 맞춘 사진 (squash)
  annotations/instances_train2017.json, instances_val2017.json
  labels.txt
"""
import json
import os
import random
import shutil
from collections import Counter
from pathlib import Path

import cv2

SRC = Path(os.environ.get("QROVER_SRC", "/content/qrover_data"))
DST = Path(os.environ.get("QROVER_DATA", "/content/YOLOX/datasets/qrover"))
SIZE = 416
VAL_SESSIONS = []      # 예: ["s05"]. 비워 두면 세션 단위로 약 20%를 자동으로 고릅니다
SEED = 0

names = [l.strip() for l in (SRC / "labels.txt").read_text(encoding="utf-8").splitlines() if l.strip()]
images = sorted(p for p in (SRC / "images").iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})


def session_of(p):
    return p.stem.split("_")[0]


sessions = sorted({session_of(p) for p in images})
assert len(sessions) >= 2, "촬영 세션이 1개뿐입니다. 파일 이름을 s01_..., s02_... 처럼 2개 이상으로 나눠 찍으세요."

if VAL_SESSIONS:
    val_sessions = set(VAL_SESSIONS)
else:
    order = sessions[:]
    random.Random(SEED).shuffle(order)
    count = Counter(session_of(p) for p in images)
    val_sessions, n = set(), 0
    for s in order:
        if n >= 0.2 * len(images):
            break
        if n + count[s] <= 0.35 * len(images):    # 너무 큰 세션은 검증으로 빼지 않음 (학습할 사진이 모자라짐)
            val_sessions.add(s)
            n += count[s]
    if not val_sessions:                          # 세션이 전부 크면 가장 작은 세션 하나
        val_sessions = {min(sessions, key=count.get)}
assert len(val_sessions) < len(sessions), "모든 세션이 검증용이 되었습니다. VAL_SESSIONS를 확인하세요."


def build(split, files):
    out = DST / f"{split}2017"
    out.mkdir(parents=True, exist_ok=True)
    coco = {
        "info": {}, "licenses": [],
        "images": [], "annotations": [],
        "categories": [{"id": i + 1, "name": n} for i, n in enumerate(names)],
    }
    boxes, background = Counter(), 0
    for img_id, p in enumerate(files, 1):
        img = cv2.imread(str(p))
        if img is None:
            print("  읽기 실패, 건너뜀:", p.name)
            continue
        img = cv2.resize(img, (SIZE, SIZE), interpolation=cv2.INTER_AREA)  # 비율 무시하고 늘려 맞춤
        cv2.imwrite(str(out / f"{p.stem}.jpg"), img)
        coco["images"].append({"id": img_id, "file_name": f"{p.stem}.jpg", "width": SIZE, "height": SIZE})

        txt = SRC / "labels" / f"{p.stem}.txt"
        lines = txt.read_text().split("\n") if txt.exists() else []
        n_obj = 0
        for line in lines:
            v = line.split()
            if len(v) != 5:
                continue
            c = int(v[0])
            assert 0 <= c < len(names), f"{txt.name}: 클래스 번호 {c}가 labels.txt 범위 밖입니다"
            cx, cy, w, h = (float(x) * SIZE for x in v[1:])
            coco["annotations"].append({
                "id": len(coco["annotations"]) + 1, "image_id": img_id, "category_id": c + 1,
                "bbox": [cx - w / 2, cy - h / 2, w, h], "area": w * h, "iscrowd": 0,
            })
            boxes[names[c]] += 1
            n_obj += 1
        background += n_obj == 0

    (DST / "annotations").mkdir(parents=True, exist_ok=True)
    with open(DST / "annotations" / f"instances_{split}2017.json", "w", encoding="utf-8") as f:
        json.dump(coco, f, ensure_ascii=False)
    print(f"[{split}] 사진 {len(coco['images'])}장 (배경 {background}장) / 박스", dict(boxes))


if DST.exists():
    shutil.rmtree(DST)
build("train", [p for p in images if session_of(p) not in val_sessions])
build("val", [p for p in images if session_of(p) in val_sessions])
shutil.copy(SRC / "labels.txt", DST / "labels.txt")
print("검증 세션:", sorted(val_sessions), "/ 클래스:", names)


In [ ]:
!python /content/prepare_dataset.py

## 4. 실험 설정 (RGB로 읽기 포함)

In [ ]:
%%writefile /content/YOLOX/exps/qrover_nano.py
#!/usr/bin/env python3
# Q-ROVER 3차시 — 팀 데이터로 YOLOX-Nano 추가 학습
import os

import torch.nn as nn

from yolox.data import COCODataset
from yolox.exp import Exp as MyExp

# 평가용 C++ 확장을 컴파일하지 않고 pycocotools로 평가합니다. (Colab 컴파일 오류 방지)
import yolox.layers as _layers

if hasattr(_layers, "COCOeval_opt"):
    del _layers.COCOeval_opt

DATA_DIR = os.environ.get("QROVER_DATA", "/content/YOLOX/datasets/qrover")


class RGBCOCODataset(COCODataset):
    """보드(Edge Impulse)는 사진을 RGB 순서로 넣어 줍니다.
    YOLOX는 cv2로 읽어 BGR 순서로 학습하므로, 그대로 두면 보드에서는 빨강과 파랑이 뒤바뀐 사진을 보게 됩니다.
    사진을 읽는 함수만 바꿔서 처음부터 RGB 순서로 배우게 합니다."""

    def load_image(self, index):
        return super().load_image(index)[:, :, ::-1].copy()  # BGR → RGB


class Exp(MyExp):
    def __init__(self):
        super().__init__()
        # 데이터 — prepare_dataset이 만든 폴더
        self.data_dir = DATA_DIR
        self.train_ann = "instances_train2017.json"
        self.val_ann = "instances_val2017.json"
        with open(os.path.join(DATA_DIR, "labels.txt"), encoding="utf-8") as f:
            self.num_classes = len([l for l in f if l.strip()])

        # 모델 크기 — 보드에 기본으로 들어 있던 YOLOX-Nano와 같습니다
        self.depth = 0.33
        self.width = 0.25
        self.input_size = (416, 416)
        self.test_size = (416, 416)
        self.random_size = (10, 20)    # 학습 중 320~640 크기를 섞어 가며 학습
        self.mosaic_scale = (0.5, 1.5)
        self.mosaic_prob = 0.5
        self.enable_mixup = False
        self.flip_prob = 0.5           # 좌/우 화살표처럼 방향이 의미 있는 클래스가 있으면 0.0

        # 학습 일정 — 사진 200장 안팎 기준
        self.max_epoch = 100
        self.warmup_epochs = 5
        self.no_aug_epochs = 15
        self.eval_interval = 5
        self.print_interval = 5
        self.data_num_workers = 2
        self.save_history_ckpt = False  # 에폭마다 체크포인트를 남기지 않음 (용량 절약)

        self.exp_name = "qrover_nano"

    def get_dataset(self, cache: bool = False, cache_type: str = "ram"):
        from yolox.data import TrainTransform

        return RGBCOCODataset(
            data_dir=self.data_dir,
            json_file=self.train_ann,
            img_size=self.input_size,
            preproc=TrainTransform(max_labels=50, flip_prob=self.flip_prob, hsv_prob=self.hsv_prob),
            cache=cache,
            cache_type=cache_type,
        )

    def get_eval_dataset(self, **kwargs):
        from yolox.data import ValTransform

        return RGBCOCODataset(
            data_dir=self.data_dir,
            json_file=self.val_ann,
            name="val2017",
            img_size=self.test_size,
            preproc=ValTransform(legacy=kwargs.get("legacy", False)),
        )

    def get_model(self, sublinear=False):
        def init_yolo(M):
            for m in M.modules():
                if isinstance(m, nn.BatchNorm2d):
                    m.eps = 1e-3
                    m.momentum = 0.03

        if "model" not in self.__dict__:
            from yolox.models import YOLOX, YOLOPAFPN, YOLOXHead
            in_channels = [256, 512, 1024]
            backbone = YOLOPAFPN(self.depth, self.width, in_channels=in_channels, act=self.act, depthwise=True)
            head = YOLOXHead(self.num_classes, self.width, in_channels=in_channels, act=self.act, depthwise=True)
            self.model = YOLOX(backbone, head)

        self.model.apply(init_yolo)
        self.model.head.initialize_biases(1e-2)
        return self.model

## 5. 학습 (10~20분)

In [ ]:
%cd /content/YOLOX
!python tools/train.py -f exps/qrover_nano.py -d 1 -b 16 --fp16 -c yolox_nano.pth

## 6. 결과

In [ ]:
%load_ext tensorboard
%tensorboard --logdir YOLOX_outputs/qrover_nano

In [ ]:
!grep "IoU=0.50 " YOLOX_outputs/qrover_nano/train_log.txt | tail -1
!grep -A 6 "per class AP" YOLOX_outputs/qrover_nano/train_log.txt | tail -7

## 7. ONNX 내보내기

In [ ]:
%%writefile /content/export_qrover.py
"""학습된 체크포인트(.pth) → ONNX
YOLOX의 tools/export_onnx.py는 최신 PyTorch에서 동작하지 않아(torch.onnx._export 삭제) 같은 일을 하는 짧은 버전을 씁니다.

  python export_qrover.py exps/qrover_nano.py YOLOX_outputs/qrover_nano/best_ckpt.pth qrover_nano.onnx
"""
import sys

import torch
from torch import nn

from yolox.exp import get_exp
from yolox.models.network_blocks import SiLU
from yolox.utils import replace_module

exp_file, ckpt_file, out_file = sys.argv[1:4]
exp = get_exp(exp_file, None)
model = exp.get_model()

ckpt = torch.load(ckpt_file, map_location="cpu", weights_only=False)  # 직접 학습한 파일이라 믿고 엽니다
model.load_state_dict(ckpt["model"] if "model" in ckpt else ckpt)
model.eval()
model = replace_module(model, nn.SiLU, SiLU)   # SiLU를 x*sigmoid(x)로 풀어서 내보냄
model.head.decode_in_inference = False         # 칸(grid) 좌표 그대로 내보냄 — Edge Impulse의 YOLOX 형식이 이것

dummy = torch.randn(1, 3, *exp.test_size)
kwargs = dict(input_names=["images"], output_names=["output"], opset_version=11)
try:
    torch.onnx.export(model, dummy, out_file, dynamo=False, **kwargs)   # PyTorch 2.5 이상
except TypeError:
    torch.onnx.export(model, dummy, out_file, **kwargs)                 # 그보다 오래된 PyTorch

with torch.no_grad():
    shape = tuple(model(dummy).shape)
print(f"저장: {out_file}  입력 (1, 3, {exp.test_size[0]}, {exp.test_size[1]})  출력 {shape}"
      f"  = (1, 칸 개수, 4+1+클래스 {exp.num_classes}개)")

In [ ]:
!python /content/export_qrover.py exps/qrover_nano.py YOLOX_outputs/qrover_nano/best_ckpt.pth /content/qrover_nano.onnx

## 8. 검증 사진으로 확인

In [ ]:
%%writefile /content/onnx_check.py
"""같은 ONNX 모델을 노트북에서 돌려 봅니다. Brick도 Edge Impulse도 없이 numpy + OpenCV + onnxruntime만으로.

  python onnx_check.py qrover_nano.onnx labels.txt                      웹캠 (1 → 0 → 2번 순서로 자동, q로 종료)
  python onnx_check.py qrover_nano.onnx labels.txt --cam 0              웹캠 번호 지정
  python onnx_check.py qrover_nano.onnx labels.txt --images 폴더 --save 결과폴더
"""
import argparse
import sys
import time
from pathlib import Path

import cv2
import numpy as np
import onnxruntime as ort


def preprocess(frame_bgr, size):
    img = cv2.resize(frame_bgr, (size, size))             # ① 비율 무시하고 늘려 맞춤 — 보드(EI)의 squash와 같음
    img = img[:, :, ::-1]                                 # ② BGR → RGB — 학습 때와 같은 순서
    x = img.transpose(2, 0, 1)[None].astype(np.float32)   # ③ (1, 3, H, W), 0~255 그대로 (정규화 없음)
    return np.ascontiguousarray(x)


def decode(out, size, strides=(8, 16, 32)):
    grids, steps = [], []
    for s in strides:
        n = size // s
        xv, yv = np.meshgrid(np.arange(n), np.arange(n))
        grids.append(np.stack((xv, yv), 2).reshape(-1, 2))
        steps.append(np.full((n * n, 1), s))
    grids, steps = np.concatenate(grids), np.concatenate(steps)
    out[:, :2] = (out[:, :2] + grids) * steps   # 칸 번호 + 칸 안의 위치 → 픽셀 좌표
    out[:, 2:4] = np.exp(out[:, 2:4]) * steps   # log 크기 → 픽셀 크기
    return out


def detect(sess, frame_bgr, names, size, conf_th, nms_th=0.45):
    out = sess.run(None, {sess.get_inputs()[0].name: preprocess(frame_bgr, size)})[0][0]
    out = decode(out.copy(), size)
    scores = out[:, 4:5] * out[:, 5:]           # "뭔가 있다" 확률 × "그게 이 클래스다" 확률
    cls, conf = scores.argmax(1), scores.max(1)
    keep = conf >= conf_th
    boxes, cls, conf = out[keep, :4], cls[keep], conf[keep]

    h, w = frame_bgr.shape[:2]
    sx, sy = w / size, h / size                 # 416x416 좌표 → 원래 사진 좌표
    xywh = [[(b[0] - b[2] / 2) * sx, (b[1] - b[3] / 2) * sy, b[2] * sx, b[3] * sy] for b in boxes]
    result = {}
    for i in np.array(cv2.dnn.NMSBoxes(xywh, conf.tolist(), conf_th, nms_th)).flatten():
        x, y, bw, bh = xywh[i]
        result.setdefault(names[cls[i]], []).append(
            {"confidence": round(float(conf[i]), 2),
             "bounding_box_xyxy": (int(x), int(y), int(x + bw), int(y + bh))})
    return result                               # 보드의 on_detect_all 콜백이 받는 것과 같은 모양


def draw(frame, result):
    for label, items in result.items():
        for it in items:
            x1, y1, x2, y2 = it["bounding_box_xyxy"]
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(frame, f"{label} {it['confidence']:.2f}", (x1, max(y1 - 6, 12)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    return frame


def open_camera(index):
    # Windows 기본(MSMF)은 C270을 여는 데 30초 넘게 걸림 → DirectShow로 엶. 맥·리눅스는 기본값
    backend = cv2.CAP_DSHOW if sys.platform == "win32" else cv2.CAP_ANY
    cap = cv2.VideoCapture(index, backend)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
    if cap.isOpened() and cap.read()[0]:
        return cap
    cap.release()
    return None


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("model")
    ap.add_argument("labels")
    ap.add_argument("--cam", type=int, default=None, help="생략하면 1 → 0 → 2 순서로 열리는 카메라를 씀")
    ap.add_argument("--images", default=None)
    ap.add_argument("--save", default=None)
    ap.add_argument("--conf", type=float, default=0.5)
    args = ap.parse_args()

    names = [l.strip() for l in Path(args.labels).read_text(encoding="utf-8").splitlines() if l.strip()]
    sess = ort.InferenceSession(args.model, providers=["CPUExecutionProvider"])
    size = sess.get_inputs()[0].shape[2]
    print("입력", sess.get_inputs()[0].shape, "/ 출력", sess.get_outputs()[0].shape, "/ 클래스", names)

    if args.images:
        save = Path(args.save) if args.save else None
        if save:
            save.mkdir(parents=True, exist_ok=True)
        times = []
        for p in sorted(Path(args.images).glob("*.jpg")):
            frame = cv2.imdecode(np.fromfile(str(p), np.uint8), cv2.IMREAD_COLOR)   # 한글 경로에서도 읽힘
            t0 = time.perf_counter()
            result = detect(sess, frame, names, size, args.conf)
            times.append(time.perf_counter() - t0)
            print(p.name, result)
            if save:
                cv2.imencode(".jpg", draw(frame, result))[1].tofile(str(save / p.name))   # 한글 경로에서도 써짐
        if times:
            print(f"평균 추론 시간 {1000 * np.mean(times):.1f} ms  (약 {1 / np.mean(times):.1f} FPS)")
        return

    cap, cam = None, None
    for i in ([args.cam] if args.cam is not None else [1, 0, 2]):
        cap = open_camera(i)
        if cap:
            cam = i
            break
    if cap is None:
        raise SystemExit("카메라를 열 수 없습니다. C270 연결을 확인하고, 다른 프로그램(줌, 카메라 앱)이 쓰고 있지 않은지 보세요.")
    print(f"카메라 {cam}번 · q로 종료. 내장 카메라 화면이 뜨면 --cam 번호를 바꿔 다시 실행하세요.")
    fps = 0.0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        t0 = time.perf_counter()
        result = detect(sess, frame, names, size, args.conf)
        fps = 0.9 * fps + 0.1 / (time.perf_counter() - t0)
        draw(frame, result)
        cv2.putText(frame, f"{fps:.1f} FPS", (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        cv2.imshow("onnx_check", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    cap.release()
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()


In [ ]:
!python /content/onnx_check.py /content/qrover_nano.onnx /content/qrover_data/labels.txt \n    --images datasets/qrover/val2017 --save /content/check

from IPython.display import Image, display
import glob
for p in sorted(glob.glob("/content/check/*.jpg"))[:8]:
    display(Image(p))

## 9. Drive에 저장 (꼭 실행)

In [ ]:
!mkdir -p /content/drive/MyDrive/qrover/out
!cp /content/qrover_nano.onnx YOLOX_outputs/qrover_nano/best_ckpt.pth YOLOX_outputs/qrover_nano/train_log.txt /content/drive/MyDrive/qrover/out/